In [10]:
import zarr
import s3fs
import numpy as np
from src.train import *

from pprint import pprint

# SageMaker
import sagemaker
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, CacheConfig
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.workflow.properties import PropertyFile
from sagemaker.pytorch.estimator import PyTorch
from sagemaker.inputs import TrainingInput
from sagemaker.model import Model
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.inputs import CreateModelInput
from sagemaker.workflow.model_step import ModelStep
from sagemaker.transformer import Transformer
from sagemaker.inputs import TransformInput
from sagemaker.workflow.steps import TransformStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet

In [11]:
%store -r 

In [12]:
# Location constants
ZARR_REF = f"{s3_project_prefix}/store"
TRAIN_REF = "train_indices.npy"
VAL_REF = "val_indices.npy"
INDICES_REF = f"{s3_project_prefix}/indices"

# Data Splitting

In [13]:
# Open the S3 filesystem
fs = s3fs.S3FileSystem()

# Open the zarr store
store = s3fs.S3Map(root=os.path.join(ZARR_REF, "data.zarr"), s3=fs, check=False)
n = zarr.open(store, mode='r')['inside_masks'].shape[0]
indices = np.arange(n)

# Setting seed for reproducibility
np.random.seed(39)
np.random.shuffle(indices)

split = int(n * 0.8)
train_idx = indices[:split]
val_idx   = indices[split:]

# Persist the split across runs
np.save(TRAIN_REF, train_idx)
np.save(VAL_REF,   val_idx)

# Push to S3
fs.put(TRAIN_REF, f"{INDICES_REF}/{TRAIN_REF}")
fs.put(VAL_REF, f"{INDICES_REF}/{VAL_REF}")

[None]

# Pipeline Parameters

In [14]:
# caching
cache_config = CacheConfig(enable_caching=True, expire_after="PT1H")

# Processing params
processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount", default_value=1
    )

processing_instance_type = ParameterString(
    name="ProcessingInstanceType", default_value="ml.g5.4xlarge"
    )

# Training Params
training_instance_count = ParameterInteger(
    name="TrainingInstanceCount", default_value=1
    )

training_instance_type = ParameterString(
    name="TrainingInstanceType", default_value="ml.g5.xlarge"
    )

# Evaluation Params
eval_instance_count = ParameterInteger(
    name="EvalInstanceCount", default_value=1
    )

eval_instance_type = ParameterString(
    name="EvalInstanceType", default_value="ml.g5.xlarge"
    )

model_package_group_name = f"InteriorBoundsModelPackage"

model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="Approved"
)

mse_threshold = ParameterFloat(name="MseThreshold", default_value=0.10)

In [15]:
model_path = f"{s3_project_prefix}/models/model_artifacts"

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session._region_name
pipeline_session = PipelineSession()

# Training Step

In [ ]:
pytorch_train = PyTorch(
    entry_point="train.py",
    source_dir="./src/",
    role=role,
    framework_version="2.1.0",
    py_version="py310",
    instance_type="ml.g5.xlarge",
    instance_count=1,
    output_path=model_path,
    sagemaker_session=pipeline_session,
    environment={"PYTHONUNBUFFERED": "1"},
    hyperparameters={
        "epochs": 6,
        "learning-rate": 0.001,
        "batch-size": 64        
    },
)

train_args = pytorch_train.fit(
    inputs={
        "indices": TrainingInput(INDICES_REF, input_mode="File"),
        "zarr": TrainingInput(ZARR_REF, input_mode="FastFile")
    }
)

step_train = TrainingStep(
    name="TrainInteriorBoundsModel",
    step_args=train_args,
    cache_config=cache_config
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


# Pipeline Execution

In [17]:
pipeline = Pipeline(
    name="InteriorBoundsPipeline",
    steps=[step_train],
    sagemaker_session=pipeline_session,
)

In [18]:
pipeline.upsert(role_arn=role)
execution = pipeline.start()
execution.wait()
print(execution.describe())

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│   1 pipeline.upsert(role_arn=role)                                                               │
│   2 execution = pipeline.start()                                                                 │
│ ❱ 3 execution.wait()                                                                             │
│   4 print(execution.describe())                                                                  │
│   5                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:938 in wait               │
│                                                                                                  │
│    935 │   │   waiter = botocore.waiter.create_waiter_with_client(                               │
│    936 │   │   │   waiter_id, model, self.sagemaker_session.sagemaker_client                     │
│    937 │   │   )                                                                                 │
│ ❱  938 │   │   waiter.wait(PipelineExecutionArn=self.arn)                                        │
│    939 │                                                                                         │
│    940 │   def result(self, step_name: str):                                                     │
│    941 │   │   """Retrieves the output of the provided step if it is a ``@step`` decorated func  │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/waiter.py:55 in wait                            │
│                                                                                                  │
│    52 │   # Waiter.wait method. This is needed to attach a docstring to the                      │
│    53 │   # method.                                                                              │
│    54 │   def wait(self, **kwargs):                                                              │
│ ❱  55 │   │   Waiter.wait(self, **kwargs)                                                        │
│    56 │                                                                                          │
│    57 │   wait.__doc__ = WaiterDocstring(                                                        │
│    58 │   │   waiter_name=waiter_name,                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/waiter.py:374 in wait                           │
│                                                                                                  │
│   371 │   │   │   │   return                                                                     │
│   372 │   │   │   if current_state == 'failure':                                                 │
│   373 │   │   │   │   reason = f'Waiter encountered a terminal failure state: {acceptor.explan   │
│ ❱ 374 │   │   │   │   raise WaiterError(                                                         │
│   375 │   │   │   │   │   name=self.name,                                                        │
│   376 │   │   │   │   │   reason=reason,                                                         │
│   377 │   │   │   │   │   last_response=response,                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
WaiterError: Waiter PipelineExecutionComplete failed: Waiter 

In [ ]:
for step in execution.list_steps():
    print(f"Step: {step['StepName']}")
    print(f"Status: {step['StepStatus']}")

    if step['StepStatus'] == 'Failed':
        print(f"Failure Reason: {step['FailureReason']}")
    print("---")

In [ ]:
for step in execution.list_steps():
    if step["StepStatus"] == "Failed":
        pprint(step)